In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(path)
from utility import *
from plot_utils import save_and_trim

# --- Config ---
exp_name = "FCT_microsoft_skewed"
file_template = "../../results/FCT_microsoft_skewed/log_{network}_{x_var}gamma_HD_{y_var}pload_seed=1.txt"
networks = ["cbb_108i", "opera_ecmp"]
x_vars = ["0.00", "0.25", "0.50", "0.75", "1.00"]   # gammas
y_vars = ["2.00", "4.00", "6.00", "8.00", "10.00"]  # loads
fct_metric = "avg"

# --- Load data ---
def compute_fct_data_2d(file_template, networks, fct_metric, x_var_name, x_var_values, y_var_name, y_var_values):
    fct_data, baseline_fct, sizes = {}, {}, []
    for idx, network in enumerate(networks):
        results = {}
        for x_var in x_var_values:
            for y_var in y_var_values:
                file_path = file_template.format(network=network, x_var=x_var, y_var=y_var)
                if not os.path.exists(file_path):
                    print(f"File not found: {file_path}. Skipping...")
                    continue
                try:
                    data = get_data_from_file(file_path)
                    fct_dict = get_FCT_from_data(data)[0 if fct_metric == "avg" else 1]
                    if idx == 0:
                        baseline_fct[(x_var, y_var)] = fct_dict
                        sizes = list(fct_dict.keys())
                    results[(x_var, y_var)] = [
                        fct_dict.get(s, None) if idx == 0
                        else (fct_dict[s] / baseline_fct[(x_var, y_var)][s] if s in fct_dict else None)
                        for s in sizes
                    ]
                except Exception as e:
                    print(f"Error processing {network} at {x_var_name}={x_var}, {y_var_name}={y_var}: {e}")

        table = {"Size": sizes}
        table.update({f"{x_var_name}={x}, {y_var_name}={y}": v for (x, y), v in results.items()})
        fct_data[network] = pd.DataFrame(table)
    return fct_data

fct_results_df = compute_fct_data_2d(file_template, networks, fct_metric, "gamma", x_vars, "load", y_vars)
df = fct_results_df["opera_ecmp"]  # normalized data (2nd network)

# --- Plot ---
exclude_set = {(0.00, 10), (0.00, 12), (0.75, 10), (0.75, 12), (1.00, 8), (1.00, 10), (1.00, 12)}

def plot_heatmap(row_df, ax):
    records = []
    for col in row_df.columns[1:]:
        g, l = col.split(", ")
        gv, lv = float(g.split("=")[1]), int(float(l.split("=")[1]))
        val = np.nan if (gv, lv) in exclude_set else row_df[col].values[0]
        records.append({"gamma": gv, "load": lv, "value": val})

    pivot = pd.DataFrame(records).pivot(index="load", columns="gamma", values="value").sort_index(ascending=False)

    cmap = sns.color_palette("Greens", as_cmap=True)
    cmap.set_bad("gray")
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap=cmap, linewidths=0.5,
                linecolor="#666666", mask=pivot.isna(), ax=ax,
                annot_kws={"fontsize": 12, "weight": "bold"},
                cbar_kws={"label": ""})

    for y, x in zip(*np.where(pivot.isna())):
        ax.text(x + 0.5, y + 0.5, "-", ha="center", va="center",
                color="white", fontsize=12, fontweight="bold")

    size_mb = int(row_df["Size"].values[0] / 1e6)
    ax.set_title(f"Flow size = {size_mb} MB", fontsize=17, pad=18)
    ax.set_xlabel("$\u03B3$", fontsize=15, labelpad=10)
    ax.set_ylabel("Load (%)", fontsize=15, labelpad=10)
    ax.tick_params(labelsize=14)
    ax.collections[0].colorbar.ax.tick_params(labelsize=12)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
plot_heatmap(df.loc[df["Size"] == 98686787], axes[0])
plot_heatmap(df.loc[df["Size"] == 223092956], axes[1])
plt.tight_layout()
save_and_trim(f"{FIG_DIR}/gamma.png", dpi=DEFAULT_DPI)
plt.show()